# Data quality

A diagnostic, not a pipeline step - run it whenever you want, before or after
any of notebooks 1-4.

**Part A** checks raw questionnaires *before* they are processed, so a
malformed file can go back to the country instead of quietly entering the
compendium.

**Part B** checks the final long files *after*, looking for figures that
contradict themselves.

Every check below exists because this project has produced that exact error:

| Check | The error it caught |
| --- | --- |
| Duplicate column headers | Libya's `2024.1` - a repeated year column that became a stray empty column |
| Whitespace in merge keys | An Algeria indicator with a trailing space, which silently lost 10 citations |
| Totals reconcile | Tunisia 2010 Female: all-ages total 285,400 against age bands summing to 4,910,200 |
| Year-on-year jumps | Morocco 2024, out by a factor of exactly 1,000 |
| Implausible values | Tunisia 2015: 55,662,100 males, in a country of ~12 million |
| Percentages sum to 100 | Shares that silently drop a category |
| Duplicate rows | The same dimensions and year carrying two different values |

Nothing here changes your data. It writes one report, `data_quality_report.xlsx`,
with a sheet per check, and prints a summary.


In [ ]:
"""
CELL: Imports and logging setup.
"""
import difflib
import logging
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium")


## Config


In [ ]:
"""
CELL: Configuration.
"""
DATA_COLLECTOR_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\DATA COLLECTOR")
TRANSLATION_DICT_PATH = DATA_COLLECTOR_PATH / "translation dict.xlsx"
COMPENDIUM_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\COMPENDIUM-ARAB SOCIETY")
QUESTIONNAIRE_PREFIX = "datacollector_received_quest_"

REPORT_PATH = COMPENDIUM_PATH / "data_quality_report.xlsx"

# Leave as None to check every chapter found, or restrict e.g. ["Population"].
CHAPTERS = None

# A year-on-year change bigger than this is reported. 10x is deliberately loose:
# real populations do not change tenfold in a year, so anything caught is a
# transcription error, a units change, or a genuine break in the series.
YEAR_JUMP_FACTOR = 10

# How far a set of percentages may stray from 100 before it is reported.
PERCENT_TOLERANCE = 1.0

# How far a reported total may stray from the sum of its parts, as a fraction.
TOTAL_TOLERANCE = 0.01


## Helpers


In [ ]:
"""
CELL: Shared helpers.
"""


def to_number(value):
    """Parse one cell into a float, or None if there is no number in it.

    These files store figures as text more often than as numbers: ' 701 956 '
    uses spaces as thousand separators, some cells use commas, some carry a
    non-breaking space, and a few hold '-' for "no data".
    """
    if pd.isna(value):
        return None
    text = str(value).replace("\xa0", " ").replace(",", "").strip()
    text = re.sub(r"\s+", "", text)
    if text in ("", "-", "--", "..", "..."):
        return None
    try:
        return float(text)
    except ValueError:
        return None


def looks_arabic(text):
    return any("\u0600" <= c <= "\u06ff" for c in str(text))


def final_files():
    """The final long files to check, as (chapter, language, path)."""
    found = []
    for language in ["EN", "AR"]:
        suffix = f"_{language}.xlsx"
        for path in sorted(COMPENDIUM_PATH.glob(f"*{suffix}")):
            chapter = path.name[: -len(suffix)]
            if CHAPTERS and chapter not in CHAPTERS:
                continue
            found.append((chapter, language, path))
    return found


def dimension_columns(table):
    """The columns that identify a row, rather than carrying its value."""
    skip = {"Value", "Source", "العدد", "المصدر"}
    return [c for c in table.columns if c not in skip]


FINDINGS = {}


def record(name, frame):
    """Keep one check's findings for the report, and say how it went."""
    FINDINGS[name] = frame
    if frame.empty:
        print(f"  OK       {name}: nothing found")
    else:
        print(f"  REVIEW   {name}: {len(frame):,} finding(s)")
    return frame


## Part A - questionnaires, before processing

These read the raw files exactly as notebook 1 does, and report structural
problems that would otherwise surface much later as odd output - or not at all.


In [ ]:
"""
CELL: check_questionnaires() - structural problems in the raw files.
"""


def check_questionnaires():
    """Reads every questionnaire sheet and reports structural problems.

    Checks, in the order they bite:
      - the two 'index' header rows exist at all (cover sheets have none, which
        is expected and reported separately as "skipped", not as a problem)
      - no duplicate column headers - Libya had '2024' and '2024.1', and the
        second became a stray empty column in the output
      - year columns are plain digits, since anything else is treated as a
        label and never unpivoted
      - no leading or trailing whitespace on the merge keys, which is what
        silently breaks the source merge
    """
    problems = []
    skipped = 0
    sheets_read = 0

    merge_keys = {"السنة", "المؤشر", "الدولة",
                  "Year", "Indicator", "Country"}

    for folder in sorted(DATA_COLLECTOR_PATH.glob(f"{QUESTIONNAIRE_PREFIX}*")):
        if not folder.is_dir():
            continue
        language = folder.name[len(QUESTIONNAIRE_PREFIX):].upper()

        for chapter_dir in sorted(p for p in folder.iterdir() if p.is_dir()):
            if CHAPTERS and chapter_dir.name not in CHAPTERS:
                continue
            for file_path in sorted(chapter_dir.glob("*.xlsx")):
                if file_path.name.startswith("~$"):
                    continue
                try:
                    xls = pd.ExcelFile(file_path, engine="openpyxl")
                except Exception as error:
                    problems.append({"language": language, "chapter": chapter_dir.name,
                                     "file": file_path.name, "sheet": "-",
                                     "problem": "cannot open the file",
                                     "detail": f"{type(error).__name__}: {error}"})
                    continue

                for sheet in xls.sheet_names:
                    try:
                        raw = pd.read_excel(xls, sheet_name=sheet, header=None, dtype=str)
                    except Exception as error:
                        problems.append({"language": language, "chapter": chapter_dir.name,
                                         "file": file_path.name, "sheet": sheet,
                                         "problem": "cannot read the sheet",
                                         "detail": f"{type(error).__name__}: {error}"})
                        continue

                    header_rows = raw.index[raw[0] == "index"].tolist()
                    if len(header_rows) < 2:
                        skipped += 1        # cover / metadata tab - expected
                        continue
                    sheets_read += 1

                    here = {"language": language, "chapter": chapter_dir.name,
                            "file": file_path.name, "sheet": sheet}

                    names = raw.iloc[header_rows[0]].dropna().astype(str).str.strip().tolist()

                    seen = {n for n in names if names.count(n) > 1}
                    if seen:
                        problems.append({**here, "problem": "duplicate column header",
                                         "detail": f"{sorted(seen)} - the second becomes a stray column"})

                    year_like = [n for n in names
                                 if re.fullmatch(r"\d{4}(\.\d+)?", n) and not n.isdigit()]
                    if year_like:
                        problems.append({**here, "problem": "year column is not a plain number",
                                         "detail": f"{year_like} - will be treated as a label, not unpivoted"})

                    if not any(n.isdigit() for n in names):
                        problems.append({**here, "problem": "no year columns",
                                         "detail": "nothing to unpivot"})

                    positions = raw.iloc[header_rows[0]]
                    for position, name in positions.items():
                        if not isinstance(name, str) or name.strip() not in merge_keys:
                            continue
                        if name != name.strip():
                            problems.append({**here, "problem": "whitespace in a merge-key header",
                                             "detail": f"{name!r} - breaks the source merge"})
                        body = raw.index[raw[0] == "1"]
                        values = raw.loc[body, position].dropna()
                        untidy = [v for v in values.unique()
                                  if isinstance(v, str) and v != v.strip()]
                        if untidy:
                            problems.append({**here, "problem": "whitespace in merge-key values",
                                             "detail": f"{len(untidy)} value(s), e.g. {untidy[0]!r}"})

    print(f"  read {sheets_read} data sheet(s); {skipped} cover/metadata sheet(s) skipped")
    return record("questionnaire structure", pd.DataFrame(problems))


## Part B - the final long files

These look for figures that contradict themselves. A number can be perfectly
well-formed and still be wrong, and these are the shapes of wrong this project
has actually produced.


In [ ]:
"""
CELL: check_totals_reconcile() - does a reported total match its own parts?
"""

TOTAL_LABELS = {
    "Age Group": "Age Total",
    "Nationality": "Nationality Total",
    "Area": "Area Total",
    "Marital status": "Marital status Total",
}


def check_totals_reconcile(table, chapter, language):
    """Compare each reported "Total" against the sum of the categories beside it.

    These files carry a total row next to its parts on several dimensions. When
    the two disagree the source contradicts itself, and no amount of careful
    processing downstream can fix it. This is the check that found Tunisia 2010
    Female reporting an all-ages total of 285,400 while its own age bands summed
    to 4,910,200.

    Only runs on the English file - the column names are looked up in English.
    """
    if language != "EN":
        return pd.DataFrame()

    findings = []
    for dimension, total_label in TOTAL_LABELS.items():
        if dimension not in table.columns:
            continue

        keys = [c for c in ["Indicator", "Country", "Year", "Sex"]
                if c in table.columns and c != dimension]
        work = table[["Value", dimension] + keys].copy()
        work["number"] = work["Value"].map(to_number)
        work = work[work["number"].notna()]
        if work.empty:
            continue

        totals = work[work[dimension] == total_label].groupby(keys)["number"].first()
        parts = (work[~work[dimension].isin([total_label, "Age unknown"])]
                 .groupby(keys)["number"].sum(min_count=1))

        shared = totals.index.intersection(parts.index)
        if not len(shared):
            continue
        comparison = pd.DataFrame({"reported_total": totals.loc[shared],
                                   "sum_of_parts": parts.loc[shared]})
        comparison = comparison[comparison["reported_total"].abs() > 0]
        gap = ((comparison["sum_of_parts"] - comparison["reported_total"]).abs()
               / comparison["reported_total"].abs())
        bad = comparison[gap > TOTAL_TOLERANCE].copy()
        if bad.empty:
            continue
        bad["off_by_%"] = (gap[gap > TOTAL_TOLERANCE] * 100).round(1)
        bad["dimension"] = dimension
        bad["chapter"] = chapter
        findings.append(bad.reset_index())

    return pd.concat(findings, ignore_index=True) if findings else pd.DataFrame()


In [ ]:
"""
CELL: the remaining output checks.
"""


def check_year_on_year_jumps(table, chapter, language):
    """Values that change by more than YEAR_JUMP_FACTOR between consecutive years.

    A real population does not change tenfold in a year, so a hit is a
    transcription error, a units change (Morocco 2024 was out by exactly 1,000),
    or a genuine break in the series worth a footnote.
    """
    if language != "EN" or "Year" not in table.columns:
        return pd.DataFrame()

    keys = [c for c in dimension_columns(table) if c != "Year"]
    work = table.copy()
    work["number"] = work["Value"].map(to_number)
    work = work[work["number"].notna() & (work["number"] != 0)]
    if work.empty:
        return pd.DataFrame()

    # One value per year first. The same dimensions and year can appear twice
    # in these files (see the conflicting-duplicates check), and without this
    # the shift below compares two rows from the SAME year against each other,
    # which is meaningless.
    work = (work.groupby(keys + ["Year"], dropna=False, observed=True)["number"]
            .first().reset_index())

    work = work.sort_values(keys + ["Year"])
    grouped = work.groupby(keys, dropna=False, observed=True)
    work["previous"] = grouped["number"].shift(1)
    work["previous_year"] = grouped["Year"].shift(1)
    work = work[work["previous"].notna() & (work["previous"] != 0)]
    # Only compare across genuinely different years.
    work = work[work["Year"] > work["previous_year"]]

    ratio = work["number"] / work["previous"]
    flagged = work[(ratio > YEAR_JUMP_FACTOR) | (ratio < 1 / YEAR_JUMP_FACTOR)].copy()
    if flagged.empty:
        return pd.DataFrame()

    flagged["times_change"] = (flagged["number"] / flagged["previous"]).round(1)
    flagged["chapter"] = chapter
    keep = ["chapter"] + [c for c in ["Indicator", "Country", "Sex", "Age Group"]
                          if c in flagged.columns]
    return flagged[keep + ["previous_year", "previous", "Year", "number", "times_change"]]


def check_implausible_values(table, chapter, language):
    """Negative counts, and percentages outside 0-100.

    An indicator is treated as a percentage if its name says so - '%',
    'percentage', 'proportion', 'rate (%)'.
    """
    if language != "EN" or "Indicator" not in table.columns:
        return pd.DataFrame()

    work = table.copy()
    work["number"] = work["Value"].map(to_number)
    work = work[work["number"].notna()]
    if work.empty:
        return pd.DataFrame()

    name = work["Indicator"].astype(str).str.lower()
    is_percentage = name.str.contains(r"%|percentage|proportion|per cent", regex=True, na=False)

    negative = work[work["number"] < 0].assign(problem="negative value")
    out_of_range = work[is_percentage & ((work["number"] < 0) | (work["number"] > 100))] \
        .assign(problem="percentage outside 0-100")

    flagged = pd.concat([negative, out_of_range], ignore_index=True).drop_duplicates()
    if flagged.empty:
        return pd.DataFrame()
    flagged["chapter"] = chapter
    keep = ["chapter", "problem"] + [c for c in ["Indicator", "Country", "Year", "Sex"]
                                     if c in flagged.columns]
    return flagged[keep + ["number"]]


def check_percentages_sum(table, chapter, language):
    """Percentage breakdowns that do not add up to 100 within their group.

    Only applied where a single dimension is clearly the thing being broken
    down, so a set of shares that silently drops a category is visible.
    """
    if language != "EN" or "Indicator" not in table.columns:
        return pd.DataFrame()

    work = table.copy()
    work["number"] = work["Value"].map(to_number)
    work = work[work["number"].notna()]
    name = work["Indicator"].astype(str).str.lower()
    work = work[name.str.contains(r"%|percentage|proportion", regex=True, na=False)]
    if work.empty:
        return pd.DataFrame()

    findings = []
    for dimension, total_label in TOTAL_LABELS.items():
        if dimension not in work.columns:
            continue
        part = work[~work[dimension].isin([total_label, "Age unknown"]) & work[dimension].notna()]
        if part.empty:
            continue
        keys = [c for c in ["Indicator", "Country", "Year", "Sex"]
                if c in part.columns and c != dimension]
        if not keys:
            continue
        sums = part.groupby(keys, dropna=False, observed=True)["number"].sum(min_count=1).dropna()
        bad = sums[(sums - 100).abs() > PERCENT_TOLERANCE]
        if bad.empty:
            continue
        frame = bad.reset_index().rename(columns={"number": "adds_up_to"})
        frame["adds_up_to"] = frame["adds_up_to"].round(1)
        frame["dimension"] = dimension
        frame["chapter"] = chapter
        findings.append(frame)

    return pd.concat(findings, ignore_index=True) if findings else pd.DataFrame()


def check_duplicate_rows(table, chapter, language):
    """The same dimensions and year appearing twice with DIFFERENT values.

    Identical duplicates are harmless noise; conflicting ones mean one of the
    two figures is wrong and there is no way to tell which downstream.
    """
    if language != "EN":
        return pd.DataFrame()

    keys = dimension_columns(table)
    if not keys:
        return pd.DataFrame()

    work = table.copy()
    work["number"] = work["Value"].map(to_number)
    work = work[work["number"].notna()]
    if work.empty:
        return pd.DataFrame()

    counts = work.groupby(keys, dropna=False, observed=True)["number"].nunique()
    conflicting = counts[counts > 1]
    if conflicting.empty:
        return pd.DataFrame()

    frame = conflicting.reset_index().rename(columns={"number": "distinct_values"})
    frame["chapter"] = chapter
    return frame.head(500)


def check_units_consistent(table, chapter, language):
    """An indicator labelled as a percentage that also holds absolute counts.

    This is the error behind most of what the totals check reports: "Population
    distribution by marital status and nationality (%)" carries real percentages
    for some countries and raw head-counts for others, so its "total" is 100 in
    one row and 29 million in another. Nothing downstream can reconcile that -
    a chart of it would be meaningless - so it is worth naming directly.
    """
    if language != "EN" or "Indicator" not in table.columns:
        return pd.DataFrame()

    work = table.copy()
    work["number"] = work["Value"].map(to_number)
    work = work[work["number"].notna()]
    name = work["Indicator"].astype(str).str.lower()
    work = work[name.str.contains(r"%|percentage|proportion", regex=True, na=False)]
    if work.empty:
        return pd.DataFrame()

    findings = []
    for indicator, group in work.groupby("Indicator", observed=True):
        looks_like_counts = group["number"] > 100
        if not looks_like_counts.any() or looks_like_counts.all():
            continue          # all percentages, or the label is simply wrong
        countries = sorted(group.loc[looks_like_counts, "Country"].dropna().unique())
        findings.append({
            "chapter": chapter,
            "Indicator": indicator,
            "rows_over_100": int(looks_like_counts.sum()),
            "rows_total": len(group),
            "largest_value": round(float(group["number"].max()), 1),
            "countries_with_counts": ", ".join(map(str, countries[:8])),
        })
    return pd.DataFrame(findings)


## Run every check


In [ ]:
"""
CELL: Run Part A and Part B, and write the report.
"""
FINDINGS.clear()

print("PART A - questionnaires")
check_questionnaires()

print("\nPART B - final long files")
files = final_files()
if not files:
    print("  no final files found - run notebooks 1-3 first")
else:
    checks = {
        "totals reconcile": check_totals_reconcile,
        "year-on-year jumps": check_year_on_year_jumps,
        "implausible values": check_implausible_values,
        "percentages sum to 100": check_percentages_sum,
        "conflicting duplicate rows": check_duplicate_rows,
        "percent indicator holds counts": check_units_consistent,
    }
    collected = {name: [] for name in checks}

    for chapter, language, path in files:
        table = pd.read_excel(path, engine="openpyxl")
        print(f"  {path.name}: {len(table):,} rows")
        for name, check in checks.items():
            try:
                result = check(table, chapter, language)
            except Exception as error:
                logger.warning(f"    {name} failed on {path.name}: "
                               f"{type(error).__name__}: {error}")
                continue
            if not result.empty:
                collected[name].append(result)

    print()
    for name, frames in collected.items():
        record(name, pd.concat(frames, ignore_index=True) if frames else pd.DataFrame())

# ---------------------------------------------------------------- the report
with pd.ExcelWriter(REPORT_PATH, engine="openpyxl") as writer:
    summary = pd.DataFrame(
        [{"check": name, "findings": len(frame)} for name, frame in FINDINGS.items()]
    ).sort_values("findings", ascending=False)
    summary.to_excel(writer, sheet_name="summary", index=False)

    for name, frame in FINDINGS.items():
        if frame.empty:
            continue
        # Excel caps sheet names at 31 characters.
        frame.head(20000).to_excel(writer, sheet_name=name[:31], index=False)

total = sum(len(f) for f in FINDINGS.values())
print("\n" + "=" * 70)
print(f"{total:,} finding(s) across {len(FINDINGS)} check(s)")
print(f"Report: {REPORT_PATH.name}")
print("=" * 70)
print(summary.to_string(index=False))


## The worst offenders

The report can be long. This prints the handful most worth acting on first -
the biggest internal contradictions and the largest year-on-year jumps, which
in this project have always turned out to be real errors rather than real
change.


In [ ]:
"""
CELL: Show the findings most worth acting on.
"""
totals = FINDINGS.get("totals reconcile", pd.DataFrame())
if not totals.empty:
    print("Biggest gaps between a reported total and the sum of its parts:")
    worst = totals.sort_values("off_by_%", ascending=False).head(10)
    columns = [c for c in ["chapter", "Country", "Year", "Sex", "dimension",
                           "reported_total", "sum_of_parts", "off_by_%"]
               if c in worst.columns]
    print(worst[columns].to_string(index=False))

jumps = FINDINGS.get("year-on-year jumps", pd.DataFrame())
if not jumps.empty:
    print("\nLargest year-on-year changes:")
    worst = jumps.reindex(jumps["times_change"].abs().sort_values(ascending=False).index).head(10)
    columns = [c for c in ["chapter", "Country", "Indicator", "previous_year",
                           "previous", "Year", "number", "times_change"]
               if c in worst.columns]
    print(worst[columns].to_string(index=False, max_colwidth=40))

if totals.empty and jumps.empty:
    print("Nothing flagged by either check.")
